[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/11.%20The%20Dynamic%20Berth%20Allocation%20Problem/P11-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 11. The Dynamic Berth Allocation Problem
## Tier 2 — The Classic Heuristic (Python Implementation)

### Goal and scope
This notebook implements an Earliest Deadline First (EDF) heuristic for the Dynamic Berth Allocation Problem (DBAP). Given vessel arrival times, desired departure times (deadlines), vessel lengths, berth capacities, and vessel–berth processing times, the algorithm constructs a feasible, non-preemptive schedule when a compatible berth is available. It reports waiting, processing, service, and deadline-delay measures, and illustrates the heuristic on a small example and randomly generated instances.

The method is a greedy heuristic, not an exact optimization algorithm. It does not guarantee a globally optimal schedule or that all deadlines will be met. In the berth-selection step, it chooses the feasible berth that gives the current vessel the earliest completion time; it does not optimize a separate aggregate objective over all vessels.

### Model assumptions
- Vessel arrivals, desired departure times, lengths, berth capacities, and processing times are known.
- A vessel can use a berth only if its length does not exceed that berth's capacity and a processing time is specified for that vessel–berth pair.
- Each berth serves at most one vessel at a time. Each vessel is assigned to at most one berth and, once service starts, is not interrupted.
- A berth's availability is represented by the completion time of the most recently assigned vessel on that berth.
- Time values use one consistent unit; the example and labels use hours. Vessel length and berth capacity use the same length unit; the example uses metres.
- The model does not represent tides, setup times, berth-to-berth movement, uncertain arrivals or processing times, or other operational constraints.

### Heuristic sequence
1. Sort vessels by desired departure time, from earliest to latest. Equal deadlines retain their input order.
2. For each vessel in that order, consider each compatible berth.
3. For each eligible berth, calculate the earliest feasible start and resulting completion time.
4. Assign the vessel to the berth with the earliest completion time, then update that berth's availability.
5. Record the schedule and calculate descriptive performance measures.

The earliest-deadline rule determines **which vessel is considered next**. The earliest-completion rule determines **which berth is selected for that vessel**. A deadline affects the priority order and the reported delay; it is not a constraint requiring completion by that time.

### Mathematical description
Let $V$ be the set of vessels and $B$ the set of berths. For vessel $i \in V$, let $a_i$ be its arrival time, $d_i$ its desired departure time, and $\ell_i$ its length. For berth $j \in B$, let $L_j$ be its capacity and $p_{ij}$ the processing time of vessel $i$ on berth $j$, when that assignment is available. Let $A_j$ denote the time berth $j$ becomes available as the heuristic proceeds.

A vessel–berth assignment is eligible when $\ell_i \leq L_j$ and $p_{ij}$ is specified. For each eligible berth, the heuristic calculates
$$
s_{ij} = \max(a_i, A_j), \qquad c_{ij} = s_{ij} + p_{ij}.
$$
Here $s_{ij}$ is the earliest feasible start time and $c_{ij}$ is the corresponding completion time. It selects an eligible berth minimizing $c_{ij}$, assigns the vessel there, and updates that berth's availability to the selected completion time. The input vessel order resolves equal deadlines; the input berth order resolves equal completion times.

The schedule is constructed in the stable order $\pi$ satisfying $d_{\pi(1)} \leq d_{\pi(2)} \leq \cdots \leq d_{\pi(|V|)}$. For each vessel, the start time on its selected berth is the maximum of its arrival time and that berth's current availability. If no berth is eligible, the heuristic leaves the vessel unassigned.

For a scheduled vessel $i$, waiting time is $w_i = s_i-a_i$, processing time is $p_i$, service time is $w_i+p_i$, and delay beyond the desired departure time is $q_i=\max(0,c_i-d_i)$. Service time here means waiting plus processing; it is not elapsed time from time zero. The code implements these quantities as `waiting_time`, `processing_time`, `service_time`, and `delay_beyond_deadline`.

Sorting takes $O(|V|\log |V|)$ time. For each vessel, the algorithm examines at most $|B|$ berths, giving $O(|V||B|)$ berth evaluation time and $O(|V|\log |V|+|V||B|)$ total time, excluding reporting and visualization. The schedule and berth state require $O(|V|+|B|)$ space. These bounds describe the heuristic's computation, not the quality of its schedule.

### Concrete example
The example contains three vessels and two berths:
- Vessel 1: arrival 0, deadline 10, length 200 m, and 5,000 TEU capacity.
- Vessel 2: arrival 2, deadline 12, length 250 m, and 8,000 TEU capacity.
- Vessel 3: arrival 4, deadline 15, length 300 m, and 12,000 TEU capacity.
- Berth 1 has a capacity of 350 m; Berth 2 has a capacity of 400 m.

The berth-specific processing times are provided in the data cell. TEU capacity is included as vessel information but does not affect this heuristic's berth eligibility or processing-time calculations.

### Interpreting the results
The schedule and metrics describe the choices made by this EDF heuristic for the supplied data. Whether a schedule meets deadlines is assessed from its reported completion times. Computation times depend on the execution environment and are not evidence of solution quality. Random scalability instances are illustrative; their results can vary between runs.

In [ ]:
# Imports used by the data model, heuristic, reporting, and visualizations.
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
@dataclass
class Vessel:
    """Input data for a vessel and its processing time on each available berth."""

    id: int
    arrival_time: int
    desired_departure: int
    length: int
    teu_capacity: int
    processing_times: Dict[int, int]

    def __repr__(self):
        return (
            f"Vessel{self.id}(arrival={self.arrival_time}, "
            f"deadline={self.desired_departure}, TEU={self.teu_capacity})"
        )


@dataclass
class Berth:
    """Input capacity and current availability of a berth."""

    id: int
    capacity: int
    available_time: int = 0

    def __repr__(self):
        return (
            f"Berth{self.id}(capacity={self.capacity}m, "
            f"available={self.available_time})"
        )


# The vessel and berth data define the complete concrete example.
vessels = [
    Vessel(1, 0, 10, 200, 5000, {1: 4, 2: 5}),
    Vessel(2, 2, 12, 250, 8000, {1: 3, 2: 4}),
    Vessel(3, 4, 15, 300, 12000, {1: 6, 2: 5}),
]

berths = [
    Berth(1, 350),
    Berth(2, 400),
]

print("Problem setup")
print("Vessels in earliest-deadline order:")
for vessel in sorted(vessels, key=lambda item: item.desired_departure):
    print(f"  {vessel}")

print("\nBerths:")
for berth in berths:
    print(f"  {berth}")

In [ ]:
class EDFDBAPSolver:
    """Construct a DBAP schedule using the earliest-deadline-first heuristic."""

    def __init__(
        self,
        vessels: List[Vessel],
        berths: List[Berth],
        verbose: bool = True,
    ):
        # Stable sorting makes the input order the tie-breaker for equal deadlines.
        self.vessels = sorted(
            vessels,
            key=lambda vessel: vessel.desired_departure,
        )
        self.initial_available_times = {
            berth.id: berth.available_time for berth in berths
        }
        # The solver owns its berth state, so solving does not alter the input data.
        self.berths = {
            berth.id: Berth(
                berth.id,
                berth.capacity,
                berth.available_time,
            )
            for berth in berths
        }
        self.verbose = verbose
        self.schedule: Dict[int, Tuple[int, int, int]] = {}
        self.solution_details: List[Dict[str, int]] = []

    def solve(self) -> Dict[int, Tuple[int, int, int]]:
        """Assign vessels in deadline order, updating berth availability."""
        # Reset state so repeated calls on the same solver construct the same schedule.
        self.schedule.clear()
        self.solution_details.clear()
        for berth_id, berth in self.berths.items():
            berth.available_time = self.initial_available_times[berth_id]

        if self.verbose:
            print("EDF heuristic")
            print("=" * 30)

        for vessel in self.vessels:
            if self.verbose:
                print(
                    f"\nProcessing Vessel {vessel.id} "
                    f"(deadline: {vessel.desired_departure})"
                )

            best_berth, start_time = self.find_best_berth(vessel)

            if best_berth is None:
                if self.verbose:
                    print(f"  No eligible berth found for Vessel {vessel.id}")
                continue

            processing_time = vessel.processing_times[best_berth]
            completion_time = start_time + processing_time
            self.berths[best_berth].available_time = completion_time

            self.schedule[vessel.id] = (
                best_berth,
                start_time,
                completion_time,
            )
            waiting_time = start_time - vessel.arrival_time
            delay_beyond_deadline = max(
                0,
                completion_time - vessel.desired_departure,
            )
            self.solution_details.append(
                {
                    "vessel_id": vessel.id,
                    "berth_id": best_berth,
                    "arrival_time": vessel.arrival_time,
                    "deadline": vessel.desired_departure,
                    "start_time": start_time,
                    "completion_time": completion_time,
                    "processing_time": processing_time,
                    "waiting_time": waiting_time,
                    "delay_beyond_deadline": delay_beyond_deadline,
                    "service_time": waiting_time + processing_time,
                }
            )

            if self.verbose:
                status = (
                    "on time"
                    if delay_beyond_deadline == 0
                    else f"late by {delay_beyond_deadline} hours"
                )
                print(f"  Assigned to Berth {best_berth}")
                print(
                    f"  Start: {start_time}, completion: {completion_time}, "
                    f"waiting: {waiting_time} hours, "
                    f"processing: {processing_time} hours"
                )
                print(f"  Deadline status: {status}")

        return self.schedule

    def find_best_berth(
        self,
        vessel: Vessel,
    ) -> Tuple[Optional[int], Optional[int]]:
        """Choose the eligible berth giving this vessel the earliest completion."""
        best_berth = None
        best_completion = float("inf")
        best_start = None

        if self.verbose:
            print("  Evaluating berth options:")

        for berth_id, berth in self.berths.items():
            # Capacity and a defined processing time jointly determine eligibility.
            if (
                vessel.length <= berth.capacity
                and berth_id in vessel.processing_times
            ):
                start_time = max(
                    vessel.arrival_time,
                    berth.available_time,
                )
                completion_time = (
                    start_time + vessel.processing_times[berth_id]
                )

                if self.verbose:
                    print(
                        f"    Berth {berth_id}: start={start_time}, "
                        f"completion={completion_time}"
                    )

                # Strict comparison preserves berth input order when completions tie.
                if completion_time < best_completion:
                    best_completion = completion_time
                    best_berth = berth_id
                    best_start = start_time
            elif self.verbose:
                print(
                    f"    Berth {berth_id}: not eligible "
                    "(capacity exceeded or processing time unavailable)"
                )

        if self.verbose and best_berth is not None:
            print(
                f"  Selected Berth {best_berth} "
                f"(earliest completion: {best_completion})"
            )

        return best_berth, best_start

    def get_performance_metrics(self) -> Dict[str, float]:
        """Summarize waiting, processing, service, and deadline-delay measures."""
        if not self.solution_details:
            return {}

        df = pd.DataFrame(self.solution_details)
        return {
            "total_service_time": df["service_time"].sum(),
            "total_waiting_time": df["waiting_time"].sum(),
            "total_processing_time": df["processing_time"].sum(),
            "total_delay_beyond_deadline": df[
                "delay_beyond_deadline"
            ].sum(),
            "avg_waiting_time": df["waiting_time"].mean(),
            "avg_processing_time": df["processing_time"].mean(),
            "max_waiting_time": df["waiting_time"].max(),
            "vessels_served": len(df),
            "vessels_delayed": int(
                (df["delay_beyond_deadline"] > 0).sum()
            ),
        }


In [ ]:
# Run the heuristic on the concrete example and report its schedule and metrics.
solver = EDFDBAPSolver(vessels, berths)

start_time = time.perf_counter()
solution = solver.solve()
computation_time_ms = (time.perf_counter() - start_time) * 1000

print("\nEDF heuristic solution")
print("=" * 40)

if solution:
    df_solution = pd.DataFrame(solver.solution_details)
    df_solution = df_solution.sort_values(
        ["start_time", "berth_id", "vessel_id"]
    )

    print("\nFinal schedule:")
    for _, row in df_solution.iterrows():
        status = (
            "ON TIME"
            if row["delay_beyond_deadline"] == 0
            else f"LATE by {row['delay_beyond_deadline']} hours"
        )
        print(
            f"Vessel {int(row['vessel_id'])}: Berth "
            f"{int(row['berth_id'])}, start {int(row['start_time'])}, "
            f"completion {int(row['completion_time'])}, "
            f"waiting {int(row['waiting_time'])} hours, "
            f"processing {int(row['processing_time'])} hours — {status}"
        )

    metrics = solver.get_performance_metrics()
    print("\nPerformance metrics:")
    print(f"Total service time: {metrics['total_service_time']} hours")
    print(f"Total waiting time: {metrics['total_waiting_time']} hours")
    print(f"Total processing time: {metrics['total_processing_time']} hours")
    print(
        "Total delay beyond deadlines: "
        f"{metrics['total_delay_beyond_deadline']} hours"
    )
    print(f"Average waiting time: {metrics['avg_waiting_time']:.2f} hours")
    print(f"Vessels served: {metrics['vessels_served']}")
    print(f"Vessels delayed: {metrics['vessels_delayed']}")
else:
    print("No vessel could be assigned to an eligible berth.")

print(f"Computation time: {computation_time_ms:.3f} milliseconds")

In [ ]:
def visualize_edf_schedule(
    solution_details,
    vessels,
    berths,
    title="EDF Heuristic Schedule",
):
    """Plot the schedule, vessel timelines, metrics, and berth utilization."""
    if not solution_details:
        print("There is no schedule to visualize.")
        return

    df_solution = pd.DataFrame(solution_details)
    metrics = {
        "total_service_time": df_solution["service_time"].sum(),
        "total_waiting_time": df_solution["waiting_time"].sum(),
        "total_processing_time": df_solution["processing_time"].sum(),
        "avg_waiting_time": df_solution["waiting_time"].mean(),
    }
    vessel_by_id = {vessel.id: vessel for vessel in vessels}
    colors = plt.get_cmap("tab20")
    vessel_color = {
        vessel.id: colors(index % 20)
        for index, vessel in enumerate(vessels)
    }

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2,
        2,
        figsize=(16, 12),
    )
    fig.suptitle(title, fontsize=16, fontweight="bold")

    # The Gantt chart displays occupied berth intervals and vessel deadlines.
    for vessel in vessels:
        vessel_rows = df_solution[
            df_solution["vessel_id"] == vessel.id
        ]
        if vessel_rows.empty:
            continue
        row = vessel_rows.iloc[0]
        ax1.barh(
            row["berth_id"],
            row["processing_time"],
            left=row["start_time"],
            color=vessel_color[vessel.id],
            alpha=0.85,
            label=f"Vessel {vessel.id} ({vessel.teu_capacity} TEU)",
        )
        ax1.text(
            row["start_time"] + row["processing_time"] / 2,
            row["berth_id"],
            f"V{vessel.id}",
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
        )
        ax1.axvline(
            vessel.desired_departure,
            color="red",
            linestyle="--",
            alpha=0.45,
        )
    ax1.set_xlabel("Time (hours)")
    ax1.set_ylabel("Berth")
    ax1.set_title("Berth schedule and deadline markers")
    ax1.set_yticks([berth.id for berth in berths])
    ax1.set_yticklabels([f"Berth {berth.id}" for berth in berths])
    ax1.grid(True, alpha=0.3)
    ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    metric_names = [
        "Total service",
        "Total waiting",
        "Total processing",
        "Average waiting",
    ]
    metric_values = [
        metrics["total_service_time"],
        metrics["total_waiting_time"],
        metrics["total_processing_time"],
        metrics["avg_waiting_time"],
    ]
    bars = ax2.bar(
        metric_names,
        metric_values,
        color=["skyblue", "orange", "seagreen", "indianred"],
        alpha=0.8,
    )
    ax2.set_title("Schedule performance measures")
    ax2.set_ylabel("Time (hours)")
    ax2.tick_params(axis="x", rotation=35)
    for bar, value in zip(bars, metric_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}",
            ha="center",
            va="bottom",
        )

    # Waiting and processing segments use the same time scale as the Gantt chart.
    for index, vessel in enumerate(vessels):
        vessel_rows = df_solution[
            df_solution["vessel_id"] == vessel.id
        ]
        if vessel_rows.empty:
            continue
        row = vessel_rows.iloc[0]
        ax3.plot(
            [vessel.arrival_time, row["start_time"]],
            [index, index],
            color="tab:blue",
            linewidth=2,
            label="Waiting" if index == 0 else None,
        )
        ax3.plot(
            [row["start_time"], row["completion_time"]],
            [index, index],
            color="tab:green",
            linewidth=4,
            label="Processing" if index == 0 else None,
        )
        ax3.scatter(
            vessel.arrival_time,
            index,
            color="tab:blue",
            label="Arrival" if index == 0 else None,
            zorder=3,
        )
        ax3.scatter(
            row["completion_time"],
            index,
            color="tab:green",
            label="Completion" if index == 0 else None,
            zorder=3,
        )
        ax3.axvline(
            vessel.desired_departure,
            color="red",
            linestyle="--",
            alpha=0.35,
        )
    ax3.set_xlabel("Time (hours)")
    ax3.set_ylabel("Vessel")
    ax3.set_title("Vessel timelines and deadline markers")
    ax3.set_yticks(range(len(vessels)))
    ax3.set_yticklabels([f"Vessel {vessel.id}" for vessel in vessels])
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    # Utilization is processing time divided by the observed schedule horizon.
    horizon = df_solution["completion_time"].max()
    berth_labels = []
    utilization_values = []
    for berth in berths:
        berth_rows = df_solution[df_solution["berth_id"] == berth.id]
        busy_time = berth_rows["processing_time"].sum()
        utilization = 100 * busy_time / horizon if horizon > 0 else 0
        berth_labels.append(f"Berth {berth.id}")
        utilization_values.append(utilization)
    bars = ax4.bar(
        berth_labels,
        utilization_values,
        color="lightcoral",
        alpha=0.8,
    )
    ax4.set_title("Berth utilization over the observed schedule horizon")
    ax4.set_ylabel("Utilization (%)")
    ax4.set_ylim(0, 100)
    for bar, value in zip(bars, utilization_values):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}%",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

    columns = [
        "vessel_id",
        "berth_id",
        "start_time",
        "completion_time",
        "waiting_time",
        "processing_time",
        "delay_beyond_deadline",
    ]
    print("\nSchedule details:")
    print(df_solution[columns].to_string(index=False))


visualize_edf_schedule(solver.solution_details, vessels, berths)

In [ ]:
def validate_edf_schedule(solution_details, vessels, berths):
    """Check assignment, arrival, capacity, processing, and berth-overlap rules."""
    vessel_by_id = {vessel.id: vessel for vessel in vessels}
    berth_by_id = {berth.id: berth for berth in berths}
    vessel_ids = [row["vessel_id"] for row in solution_details]

    assert len(vessel_ids) == len(set(vessel_ids)), (
        "A vessel appears more than once in the schedule."
    )

    for row in solution_details:
        vessel = vessel_by_id[row["vessel_id"]]
        berth = berth_by_id[row["berth_id"]]
        assert vessel.length <= berth.capacity
        assert row["berth_id"] in vessel.processing_times
        assert row["start_time"] >= vessel.arrival_time
        assert row["completion_time"] == (
            row["start_time"] + row["processing_time"]
        )
        assert row["processing_time"] == vessel.processing_times[row["berth_id"]]
        assert row["waiting_time"] == row["start_time"] - vessel.arrival_time
        assert row["service_time"] == (
            row["waiting_time"] + row["processing_time"]
        )

    for berth_id in berth_by_id:
        berth_rows = sorted(
            (
                row
                for row in solution_details
                if row["berth_id"] == berth_id
            ),
            key=lambda row: row["start_time"],
        )
        for previous, following in zip(berth_rows, berth_rows[1:]):
            assert previous["completion_time"] <= following["start_time"], (
                f"Overlapping services are scheduled on Berth {berth_id}."
            )

    assigned_ids = set(vessel_ids)
    unassigned_ids = sorted(set(vessel_by_id) - assigned_ids)
    print("Schedule feasibility checks passed for all assigned vessels.")
    if unassigned_ids:
        print(
            "Vessels without an eligible berth in the constructed schedule: "
            f"{unassigned_ids}"
        )
    else:
        print("Every vessel was assigned to a berth.")


validate_edf_schedule(solver.solution_details, vessels, berths)

In [ ]:
def scalability_analysis():
    """Illustrate EDF runtime and schedule measures on generated instances."""
    print("Scalability analysis")
    print("=" * 30)

    def generate_problem(n_vessels, n_berths):
        """Generate one illustrative instance using the standard random module."""
        generated_vessels = []
        for vessel_id in range(1, n_vessels + 1):
            arrival_time = random.randint(0, 20)
            deadline = arrival_time + random.randint(8, 20)
            length = random.randint(150, 350)
            teu_capacity = random.randint(3000, 15000)
            processing_times = {
                berth_id: random.randint(3, 10)
                for berth_id in range(1, n_berths + 1)
            }
            generated_vessels.append(
                Vessel(
                    vessel_id,
                    arrival_time,
                    deadline,
                    length,
                    teu_capacity,
                    processing_times,
                )
            )

        generated_berths = [
            Berth(berth_id, random.randint(300, 400))
            for berth_id in range(1, n_berths + 1)
        ]
        return generated_vessels, generated_berths

    test_sizes = [(5, 2), (10, 3), (15, 4), (20, 5), (25, 6), (30, 8)]
    results = []

    for n_vessels, n_berths in test_sizes:
        print(f"\nTesting {n_vessels} vessels and {n_berths} berths")
        test_vessels, test_berths = generate_problem(n_vessels, n_berths)
        test_solver = EDFDBAPSolver(
            test_vessels,
            test_berths,
            verbose=False,
        )

        start_time = time.perf_counter()
        test_solver.solve()
        computation_time_ms = (time.perf_counter() - start_time) * 1000
        metrics = test_solver.get_performance_metrics()

        if metrics:
            results.append(
                [
                    n_vessels,
                    n_berths,
                    computation_time_ms,
                    metrics["total_service_time"],
                    metrics["total_waiting_time"],
                    metrics["vessels_served"],
                    metrics["vessels_delayed"],
                ]
            )
            print(f"  Vessels served: {metrics['vessels_served']}/{n_vessels}")
            print(
                "  Total service time: "
                f"{metrics['total_service_time']:.1f} hours"
            )
        else:
            print("  No vessel could be assigned to an eligible berth.")

    if not results:
        print("No generated instance produced an assignment.")
        return

    columns = [
        "Vessels",
        "Berths",
        "Computation time (ms)",
        "Service time (hours)",
        "Waiting time (hours)",
        "Vessels served",
        "Vessels delayed",
    ]
    scalability_df = pd.DataFrame(results, columns=columns)
    print("\nScalability results:")
    print(scalability_df.to_string(index=False, float_format="%.2f"))

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2,
        2,
        figsize=(15, 10),
    )
    fig.suptitle("EDF scalability analysis", fontsize=16, fontweight="bold")

    ax1.plot(
        [row[0] for row in results],
        [row[2] for row in results],
        "bo-",
    )
    ax1.set_xlabel("Number of vessels")
    ax1.set_ylabel("Computation time (ms)")
    ax1.set_title("Computation time")
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        [row[0] for row in results],
        [row[3] for row in results],
        "ro-",
    )
    ax2.set_xlabel("Number of vessels")
    ax2.set_ylabel("Total service time (hours)")
    ax2.set_title("Constructed-schedule service time")
    ax2.grid(True, alpha=0.3)

    success_rates = [row[5] / row[0] * 100 for row in results]
    ax3.bar(range(len(results)), success_rates, color="seagreen", alpha=0.8)
    ax3.set_xlabel("Generated instance")
    ax3.set_ylabel("Vessels served (%)")
    ax3.set_title("Assignment rate")
    ax3.set_xticks(range(len(results)))
    ax3.set_xticklabels(
        [f"{row[0]} vessels, {row[1]} berths" for row in results],
        rotation=35,
        ha="right",
    )
    ax3.grid(True, alpha=0.3)

    delay_rates = [
        row[6] / row[5] * 100 if row[5] else 0
        for row in results
    ]
    ax4.bar(range(len(results)), delay_rates, color="indianred", alpha=0.8)
    ax4.set_xlabel("Generated instance")
    ax4.set_ylabel("Served vessels delayed (%)")
    ax4.set_title("Deadline-delay rate")
    ax4.set_xticks(range(len(results)))
    ax4.set_xticklabels(
        [f"{row[0]} vessels, {row[1]} berths" for row in results],
        rotation=35,
        ha="right",
    )
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


scalability_analysis()

## Key insights from the EDF heuristic

### How to read the method
- **Priority rule:** vessels are processed in nondecreasing order of desired departure time. This is the earliest-deadline-first part of the procedure.
- **Berth rule:** for the current vessel, the algorithm evaluates eligible berths using their current availability and selects the one with the earliest resulting completion.
- **State update:** after an assignment, only the selected berth's availability changes. This records the non-overlap of services on that berth.
- **Deadline measure:** the deadline is a priority and a reporting reference. The heuristic can produce a completion after a deadline because it does not impose deadline feasibility as a hard constraint.
- **Schedule measures:** total service time is total waiting plus total processing time. Total deadline delay is reported separately and is not part of service time.

### Strengths and limitations
The method is straightforward to trace and implement. Its sorting and berth scans have polynomial computational cost, making the procedure suitable for quickly constructing schedules for the modeled data. For a fixed input order and fixed data, its tie-breaking rules make the result deterministic.

Its greedy decisions are irrevocable: a berth selected for one vessel is not reconsidered when later vessels are processed. The rule uses the currently known vessels and their data but does not look ahead to compare the consequences of alternative earlier assignments. It therefore provides no general guarantee of global optimality, deadline feasibility, or a particular percentage gap from an optimum. A vessel with no eligible berth is left unassigned.

### Interpreting the experiments
The small example makes the priority and berth-selection steps traceable. The validation cell checks that every assigned vessel respects arrival and berth-capacity conditions, uses a defined processing time, and does not overlap another service on the same berth. It also checks that the reported service-time components agree with their definitions.

The scalability experiment generates new random data when run; it does not set a random seed. Its instance values and measured runtimes can therefore vary across executions and hardware. The experiment illustrates execution and reporting behavior, not a guarantee about schedule quality or performance on all DBAP instances.